# CliffForge — ChemProp Baseline on Dablander MMP Dataset

**Purpose**: Train ChemProp (absolute pIC50 GNN) on the same 1,924-molecule Dablander MMP split used
by `CliffForge_MMP_Colab.ipynb`, evaluate with identical metrics, then merge with the existing
`summary_mmp.json` to produce a **combined three-model comparison**:
- CliffForge (Siamese D-MPNN) — from previous notebook
- 5-NN baseline — from previous notebook
- **ChemProp absolute** — this notebook

**Steps:**
1. Install dependencies (fast — reuse Colab PyTorch)
2. Clone CliffForge repo
3. Download & pre-process Dablander data (identical split, SEED=17)
4. Train ChemProp on the train partition
5. Predict pIC50 for all molecules
6. Evaluate: direction accuracy, cliff AP/F1/ROC-AUC, within-series Spearman ρ
7. Upload `summary_mmp.json` from the first notebook → merge → download combined JSON

> ⚡ **Enable T4 GPU**: *Runtime → Change runtime type → T4 GPU*  
> 🕐 **Estimated time**: ~20 min (ChemProp trains on CPU; GPU not used by ChemProp CLI)

## 0 · Environment Check

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode==0 else 'None detected')
print('Python:', sys.version)
import torch
print(f'torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')


## 1 · Install Dependencies

> ⚡ Fast install (~3–4 min).
>
> ⚠️ **ChemProp 1.6.1 requires numpy < 2.0** — Colab ships with numpy 2.x, so we must downgrade it first.
> The install cell **automatically restarts the runtime** at the end to reload numpy 1.26.4.
> After the restart, **skip the install cell and run from §0 downward**.

In [ ]:
import torch, subprocess, sys

torch_base = torch.__version__.split('+')[0]
cuda_str   = (torch.version.cuda or '').replace('.', '')
pyg_url    = f'https://data.pyg.org/whl/torch-{torch_base}+cu{cuda_str}.html'
print(f'torch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'PyG index: {pyg_url}\n')

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# ── Step 1: numpy 1.26.4 FIRST (chemprop 1.x uses numpy.VisibleDeprecationWarning,
#            which was removed in numpy 2.0; Colab ships with numpy 2.x) ─────────────
print('Pinning numpy==1.26.4 (required by chemprop 1.6.1)...')
pip('numpy==1.26.4')

# ── Step 2: ChemProp (installs AFTER numpy is pinned) ────────────────────────────────
print('Installing chemprop==1.6.1...')
pip('chemprop==1.6.1')

# ── Step 3: PyG pre-built wheels ─────────────────────────────────────────────────────
print('Installing PyG wheels...')
pip('torch-scatter', 'torch-sparse', 'torch-cluster',
    'torch-spline-conv', 'torch-geometric', '-f', pyg_url)

# ── Step 4: RDKit + extras ───────────────────────────────────────────────────────────
pip('rdkit')
pip('scikit-learn', 'scipy')

# ── Step 5: Restart runtime so Python reloads numpy 1.26.4 instead of 2.x ───────────
print('\n✓ Install done. Restarting runtime to reload numpy 1.26.4 ...')
print('  → After restart, START FROM THE NEXT CELL (§0). Do NOT re-run this cell.')
import os
os.kill(os.getpid(), 9)


In [ ]:
# ── Run this AFTER the auto-restart (skip the install cell above) ───────────────
import importlib
for pkg in ['numpy','scipy','pandas','sklearn','rdkit','torch','torch_geometric','chemprop']:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', '?')
        flag = ' ⚠ (need <2.0)' if pkg=='numpy' and ver.startswith('2') else ''
        print(f'  OK  {pkg} {ver}{flag}')
    except ImportError as e:
        print(f'  MISSING {pkg}: {e}')

import numpy as np
assert np.__version__ < '2', (
    f'numpy {np.__version__} is too new for chemprop 1.6.1. '
    'Re-run the install cell from a fresh session.')
print('\nnumpy version OK for chemprop.')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2 · Clone CliffForge Repository

In [ ]:
import os, sys

REPO = '/content/TELEMEDICON26'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/chidwipak/TELEMEDICON26.git {REPO}
else:
    print('Already cloned.')

src = os.path.join(REPO, 'src')
if src not in sys.path:
    sys.path.insert(0, src)

from cliffforge import metrics as M
print('cliffforge.metrics imported OK')


## 3 · Download & Pre-Process Data

**Identical preprocessing to `CliffForge_MMP_Colab.ipynb`** — same seed, same split,
same cliff definition — guaranteeing a fair comparison.

In [ ]:
import os, math
import numpy as np
import pandas as pd
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

os.makedirs('/content/mmp_data/raw', exist_ok=True)
BASE = 'https://raw.githubusercontent.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments/main/data/postera_sars_cov_2_mpro'
!wget -q -O /content/mmp_data/raw/molecule_data_clean.csv {BASE}/molecule_data_clean.csv
!wget -q -O /content/mmp_data/raw/MMP_data_clean.csv      {BASE}/MMP_data_clean.csv

mol_raw = pd.read_csv('/content/mmp_data/raw/molecule_data_clean.csv')
mmp_raw = pd.read_csv('/content/mmp_data/raw/MMP_data_clean.csv')
print(f'Molecules: {len(mol_raw):,}  |  MMP pairs: {len(mmp_raw):,}')


In [ ]:
# ── Canonicalize + pIC50 conversion ──────────────────────────────────────────
def canon2d(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None, None
    flat = Chem.MolToSmiles(m, isomericSmiles=False)
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(flat))
        scaf_smi = Chem.MolToSmiles(scaf)
    except Exception:
        scaf_smi = 'NONE'
    return flat, scaf_smi

records = []
for _, row in mol_raw.iterrows():
    ic50 = float(row['f_avg_IC50 [uM]'])
    if ic50 <= 0: continue
    flat, scaf = canon2d(row['SMILES'])
    if flat is None: continue
    records.append({'smiles': flat, 'pIC50': 6.0 - math.log10(ic50), 'scaf': scaf})

by_smi = defaultdict(list)
for r in records: by_smi[r['smiles']].append(r['pIC50'])
deduped = [{'smiles': s, 'pIC50': float(np.mean(v)),
             'scaf': next(r['scaf'] for r in records if r['smiles']==s), 'n_rep': len(v)}
           for s,v in by_smi.items()]
print(f'After canon+dedup: {len(deduped):,} molecules')

# ── Scaffold-aware split (SEED=17, identical to first notebook) ───────────────
MIN_SERIES, TEST_FRAC, SPLIT_SEED = 8, 0.35, 17
rng = np.random.RandomState(SPLIT_SEED)
scaf_members = defaultdict(list)
for rec in deduped: scaf_members[rec['scaf']].append(rec)

for scaf, members in scaf_members.items():
    sz = len(members)
    if sz >= MIN_SERIES:
        idx = rng.permutation(sz)
        n_test = min(max(3, int(round(sz*TEST_FRAC))), sz-3)
        test_ids = set(idx[:n_test].tolist())
        for i, rec in enumerate(members):
            rec['series'], rec['series_size'] = scaf, sz
            rec['split'] = 'test' if i in test_ids else 'train'
    else:
        for rec in members:
            rec['series'] = scaf if sz>1 else 'singleton'
            rec['series_size'] = sz
            rec['split'] = 'train'

df_all = pd.DataFrame(deduped)[['smiles','pIC50','series','series_size','split','n_rep']]
n_train = int((df_all.split=='train').sum())
n_test  = int((df_all.split=='test').sum())
print(f'Train: {n_train:,}  Test: {n_test:,}  Series with test: {df_all[df_all.split=="test"]["series"].nunique()}')


In [ ]:
# ── Classify MMP pairs + compute Tanimoto + cliff flags ───────────────────────
smi_to_split  = dict(zip(df_all.smiles, df_all.split))
smi_to_pic50  = dict(zip(df_all.smiles, df_all.pIC50))
smi_to_series = dict(zip(df_all.smiles, df_all.series))

def canon_flat(smi):
    m = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(m, isomericSmiles=False) if m else None

CLIFF_SIM, CLIFF_DELTA = 0.80, 1.0

mmp_rows, skipped = [], 0
for _, row in mmp_raw.iterrows():
    s1, s2 = canon_flat(row['smiles_1']), canon_flat(row['smiles_2'])
    if not s1 or not s2 or s1==s2: skipped+=1; continue
    if s1 not in smi_to_split or s2 not in smi_to_split: skipped+=1; continue
    sp1, sp2 = smi_to_split[s1], smi_to_split[s2]
    pt = ('train_train' if sp1=='train' and sp2=='train' else
          'test_test'   if sp1=='test'  and sp2=='test'  else 'test_train')
    mmp_rows.append({'smiles_1':s1,'smiles_2':s2,
                     'pIC50_1':smi_to_pic50[s1],'pIC50_2':smi_to_pic50[s2],
                     'delta_pIC50':smi_to_pic50[s1]-smi_to_pic50[s2],
                     'abs_delta':abs(smi_to_pic50[s1]-smi_to_pic50[s2]),
                     'split_1':sp1,'split_2':sp2,'pair_type':pt,
                     'series_1':smi_to_series[s1],'series_2':smi_to_series[s2],
                     'same_series':smi_to_series[s1]==smi_to_series[s2]})
df_mmp = pd.DataFrame(mmp_rows)

# Tanimoto
print('Computing fingerprints...')
smi_list = list(df_all.smiles)
fps      = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s),2,2048) for s in smi_list]
smi_to_fp = dict(zip(smi_list, fps))
tans = [DataStructs.TanimotoSimilarity(smi_to_fp[r.smiles_1], smi_to_fp[r.smiles_2])
        for r in df_mmp.itertuples()]
df_mmp['tanimoto'] = tans
df_mmp['is_cliff'] = (df_mmp.tanimoto>=CLIFF_SIM)&(df_mmp.abs_delta>=CLIFF_DELTA)

for pt in ['train_train','test_train','test_test']:
    s = df_mmp[df_mmp.pair_type==pt]
    print(f'  {pt}: pairs={len(s):,}  cliffs={s.is_cliff.sum():,}')

# Save processed files (needed by ChemProp CLI)
os.makedirs('/content/mmp_data/processed', exist_ok=True)
df_all.to_csv('/content/mmp_data/processed/all.csv', index=False)
df_all[df_all.split=='train'][['smiles','pIC50']].to_csv('/content/mmp_data/processed/train.csv', index=False)
df_all[df_all.split=='test'] [['smiles','pIC50']].to_csv('/content/mmp_data/processed/test.csv',  index=False)
print('Processed files saved.')


## 4 · Train ChemProp (Absolute pIC50 GNN)

ChemProp uses its own directed-MPNN architecture trained to predict a single
pIC50 value per molecule. We use the same hyperparameters as the original prototype
(60 epochs, regression, RMSE metric). ChemProp trains on CPU even with GPU available.

> ⏱ Estimated: **10–15 min** (60 epochs × 1,582 molecules on CPU)

In [ ]:
import subprocess, shutil

CHEMPROP_MODEL_DIR = '/content/chemprop_mmp'
os.makedirs(CHEMPROP_MODEL_DIR, exist_ok=True)

# Locate the chemprop_train executable
cp_train = shutil.which('chemprop_train') or (sys.executable.replace('python','chemprop_train'))
print(f'chemprop_train: {cp_train}')

train_csv = '/content/mmp_data/processed/train.csv'
print(f'Training on {n_train:,} molecules for 60 epochs (CPU)...')

result = subprocess.run([
    cp_train,
    '--data_path',     train_csv,
    '--dataset_type',  'regression',
    '--smiles_columns','smiles',
    '--target_columns','pIC50',
    '--save_dir',      CHEMPROP_MODEL_DIR,
    '--epochs',        '60',
    '--num_folds',     '1',
    '--metric',        'rmse',
    '--extra_metrics', 'r2',
    '--no_cuda',
    '--num_workers',   '0',
    '--quiet',
], capture_output=True, text=True)

if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
    raise RuntimeError('ChemProp training failed. See error above.')

print('Training complete. Output:')
# Extract validation RMSE from stdout
for line in result.stdout.splitlines():
    if 'rmse' in line.lower() or 'r2' in line.lower():
        print(' ', line)


## 5 · Predict pIC50 for All Molecules

In [ ]:
PREDS_CSV = '/content/chemprop_mmp_preds.csv'
ALL_SMILES_CSV = '/content/mmp_data/processed/all_smiles.csv'

# ChemProp predict on all molecules (train + test)
df_all[['smiles']].to_csv(ALL_SMILES_CSV, index=False)

cp_predict = shutil.which('chemprop_predict') or cp_train.replace('train','predict')
print(f'chemprop_predict: {cp_predict}')
print(f'Predicting for {len(df_all):,} molecules...')

result2 = subprocess.run([
    cp_predict,
    '--test_path',      ALL_SMILES_CSV,
    '--checkpoint_dir', CHEMPROP_MODEL_DIR,
    '--preds_path',     PREDS_CSV,
    '--no_cuda',
], capture_output=True, text=True)

if result2.returncode != 0:
    print('STDERR:', result2.stderr[-3000:])
    raise RuntimeError('ChemProp prediction failed.')

preds_df = pd.read_csv(PREDS_CSV)
print(f'Predictions shape: {preds_df.shape}')
print(preds_df.head(3))

# Build lookup: smiles → predicted pIC50
pred_col_name = [c for c in preds_df.columns if c != 'smiles'][0]
smi_to_pred_cp = dict(zip(preds_df['smiles'], preds_df[pred_col_name]))

# Sanity check
covered = sum(1 for s in df_all.smiles if s in smi_to_pred_cp)
print(f'Coverage: {covered}/{len(df_all)} molecules have predictions')


In [ ]:
# ── Add ChemProp delta predictions to MMP pair table ─────────────────────────
df_eval = df_mmp[df_mmp.pair_type != 'train_train'].copy().reset_index(drop=True)

df_eval['pred_cp_1'] = [smi_to_pred_cp.get(s, np.nan) for s in df_eval.smiles_1]
df_eval['pred_cp_2'] = [smi_to_pred_cp.get(s, np.nan) for s in df_eval.smiles_2]
df_eval['pred_chemprop_delta'] = df_eval['pred_cp_1'] - df_eval['pred_cp_2']

n_valid = df_eval['pred_chemprop_delta'].notna().sum()
print(f'Valid ChemProp pair predictions: {n_valid:,} / {len(df_eval):,}')

# Absolute pIC50 predictions per molecule
cp_pred_all = np.array([smi_to_pred_cp.get(s, np.nan) for s in df_all.smiles])

# Quick RMSE on test set
test_mask = (df_all.split=='test').values
y_test = df_all.pIC50.values[test_mask]
p_test = cp_pred_all[test_mask]
valid  = np.isfinite(p_test)
rmse   = float(np.sqrt(np.mean((y_test[valid]-p_test[valid])**2)))
corr   = float(np.corrcoef(y_test[valid], p_test[valid])[0,1])
print(f'Test RMSE: {rmse:.4f} pIC50 units  |  Pearson r: {corr:.4f}')


## 6 · Evaluate ChemProp

Using `cliffforge.metrics` — **identical functions** to those used in the MMP notebook,
ensuring a fair, apples-to-apples comparison.

In [ ]:
from cliffforge import metrics as M

dir_results_cp = {}
print(f'  {"Subset":<40} {"n":>6} {"DirAcc":>8} {"p":>10}')
print('  ' + '─'*70)

for pt in ['test_test','test_train']:
    sub = df_eval[df_eval.pair_type==pt].dropna(subset=['pred_chemprop_delta'])
    for min_d, lbl in [(0.0,'all'),(0.4,'|d|>=0.4'),(1.0,'cliff(|d|>=1.0)')]:
        s2 = sub[sub.abs_delta>=min_d]
        if len(s2) < 3: continue
        key = f'{pt}|{lbl}'
        r = M.direction_accuracy(s2.delta_pIC50.values, s2.pred_chemprop_delta.values)
        dir_results_cp[key] = r
        if r['direction_accuracy'] is not None:
            p = r['binom_p_vs_0.5']
            sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
            print(f'  {key:<40} {r["n"]:>6} {r["direction_accuracy"]:>8.3f} {p:>10.4f} {sig}')

print()
print('Direction accuracy evaluation done.')


In [ ]:
cliff_results_cp = {}
TAU = 1.0

print(f'Cliff detection (tan>={CLIFF_SIM}, |Δ|>={CLIFF_DELTA}, tau={TAU})\n')
for pt in ['test_test','test_train']:
    sub = df_eval[(df_eval.pair_type==pt)&(df_eval.tanimoto>=CLIFF_SIM)].dropna(subset=['pred_chemprop_delta'])
    if len(sub)==0: continue
    n_cl = sub.is_cliff.sum()
    print(f'── {pt}  close={len(sub):,}  cliffs={n_cl}')
    m = M.cliff_detection_metrics(sub.is_cliff.values, np.abs(sub.pred_chemprop_delta.values), TAU)
    cliff_results_cp[pt] = m
    ap_s = f"{m.get('ap',float('nan')):.3f}" if 'ap' in m else 'N/A'
    print(f'  chemprop_abs: AP={ap_s}  F1={m["f1"]:.3f}  '
          f'P={m["precision"]:.3f}  R={m["recall"]:.3f}  tp={m["tp"]} fn={m["fn"]}')

print()
print('Cliff detection done.')


In [ ]:
# ── Within-series Spearman ρ ───────────────────────────────────────────────────
from scipy.stats import wilcoxon

test_mask_bool = (df_all.split=='test').values
y_all    = df_all.pIC50.values.astype(float)
series_a = df_all.series.values

# ChemProp absolute prediction → within-series Spearman directly
ws_cp = M.within_series_spearman(
    true_y    = y_all[test_mask_bool],
    pred_y    = cp_pred_all[test_mask_bool],
    series_ids= series_a[test_mask_bool],
    min_series_size=5,
)

print(f'Within-series Spearman ρ (ChemProp):')
print(f'  Series: {ws_cp["n_series"]}')
print(f'  Mean ρ: {ws_cp["mean_rho"]:.4f}')
print(f'  Per series: {[",".join(f"{x:.2f}" for x in ws_cp["per_series"])]}')


## 7 · Combine with Existing MMP Results

Upload `summary_mmp.json` from the **first notebook** (`CliffForge_MMP_Colab.ipynb`).
It should be in your downloads folder.
After uploading, ChemProp results will be merged in and a combined JSON saved.

In [ ]:
from google.colab import files as colab_files
import json

print('Upload your summary_mmp.json from the first notebook:')
uploaded = colab_files.upload()   # browser file picker

# Load whichever file was uploaded
fname = list(uploaded.keys())[0]
base_summary = json.loads(uploaded[fname])
print(f'Loaded: {fname}')
print(f'Keys in base summary: {list(base_summary.keys())}')


In [ ]:
import copy, json

def safe_json(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(f'{type(o)}')

# Build ChemProp result block matching the MMP notebook's format
chemprop_block = {
    'cliff_detection': {
        'test_test|chemprop_abs' : cliff_results_cp.get('test_test', {}),
        'test_train|chemprop_abs': cliff_results_cp.get('test_train', {}),
    },
    'direction_accuracy': dir_results_cp,
    'within_series_spearman': ws_cp,
    'test_rmse_pIC50'   : rmse,
    'test_pearson_r'    : corr,
    'training_epochs'   : 60,
    'checkpoint_dir'    : CHEMPROP_MODEL_DIR,
}

# Deep-merge into base summary
combined = copy.deepcopy(base_summary)

# Add ChemProp to cliff_detection
combined.setdefault('cliff_detection', {}).update(chemprop_block['cliff_detection'])

# Add ChemProp to direction_accuracy (per subset)
for subset, res in dir_results_cp.items():
    combined.setdefault('direction_accuracy', {}).setdefault(subset, {})['chemprop_abs'] = res

# Add within-series Spearman
combined.setdefault('within_series_spearman', {})['chemprop_abs_mean'] = ws_cp['mean_rho']
combined.setdefault('within_series_spearman', {})['chemprop_abs_per_series'] = ws_cp['per_series']
combined.setdefault('within_series_spearman', {})['chemprop_abs_n_series'] = ws_cp['n_series']

# Metadata
combined['chemprop_abs_meta'] = {
    'test_rmse': rmse, 'test_pearson_r': corr,
    'training_epochs': 60, 'n_train': n_train
}

os.makedirs('/content/results_mmp', exist_ok=True)
out_path = '/content/results_mmp/summary_mmp_combined.json'
with open(out_path, 'w') as f:
    json.dump(combined, f, indent=2, default=safe_json)
print(f'Combined summary saved → {out_path}')
print(f'File size: {os.path.getsize(out_path):,} bytes')


## 8 · Full Comparison Table

In [ ]:
import json

# ── Helper to pull from combined dict ────────────────────────────────────────
def get_ap(combined, setting_model_key):
    return combined.get('cliff_detection',{}).get(setting_model_key,{}).get('ap', float('nan'))

def get_f1(combined, setting_model_key):
    return combined.get('cliff_detection',{}).get(setting_model_key,{}).get('f1', float('nan'))

def get_da(combined, subset, model):
    return combined.get('direction_accuracy',{}).get(subset,{}).get(model,{}).get('da', float('nan'))

# Setting keys in the combined dict
# CliffForge uses 'test_test|rel_siamese', 5-NN uses 'test_test|knn_5nn',
# ChemProp uses 'test_test|chemprop_abs'

ws = combined.get('within_series_spearman', {})

print('=' * 90)
print('  FULL THREE-MODEL COMPARISON: Dablander MMP Dataset (1,924 molecules)')
print('=' * 90)
print(f'{"Metric":<50} {"CliffForge":>12} {"ChemProp":>12} {"5-NN":>10}')
print('─' * 90)

rows = [
    # Within-series ranking
    ('Within-series Spearman ρ (test, ≥5 members)',
     ws.get('rel_mean',    float('nan')),
     ws.get('chemprop_abs_mean', float('nan')),
     ws.get('knn_mean',   float('nan'))),

    # Direction accuracy — cliff pairs
    ('Direction acc, test-test, cliff (n=63)',
     get_da(combined, 'test_test|cliff(|d|>=1.0)', 'rel_siamese'),
     get_da(combined, 'test_test|cliff(|d|>=1.0)', 'chemprop_abs'),
     get_da(combined, 'test_test|cliff(|d|>=1.0)', 'knn_5nn')),

    # Cliff detection — test-test
    ('AP, test-test (strict, 2 cliffs)',
     get_ap(combined, 'test_test|rel_siamese'),
     get_ap(combined, 'test_test|chemprop_abs'),
     get_ap(combined, 'test_test|knn_5nn')),

    ('F1, test-test (strict)',
     get_f1(combined, 'test_test|rel_siamese'),
     get_f1(combined, 'test_test|chemprop_abs'),
     get_f1(combined, 'test_test|knn_5nn')),

    # Cliff detection — test-train (lead-opt)
    ('AP, test-train / lead-opt (25 cliffs)',
     get_ap(combined, 'test_train|rel_siamese'),
     get_ap(combined, 'test_train|chemprop_abs'),
     get_ap(combined, 'test_train|knn_5nn')),

    ('F1, test-train / lead-opt',
     get_f1(combined, 'test_train|rel_siamese'),
     get_f1(combined, 'test_train|chemprop_abs'),
     get_f1(combined, 'test_train|knn_5nn')),
]

for name, cf, cp, knn in rows:
    def fmt(v):
        return f'{v:.3f}' if np.isfinite(v) else 'N/A'
    # Mark best
    vals = [v for v in [cf, cp, knn] if np.isfinite(v)]
    best = max(vals) if vals else None
    def fmt_b(v):
        s = fmt(v)
        return f'{s} ★' if (np.isfinite(v) and v==best) else f'{s}  '
    print(f'{name:<50} {fmt_b(cf):>12} {fmt_b(cp):>12} {fmt(knn):>10}')

print('─' * 90)
print(f'ChemProp test RMSE: {rmse:.4f} pIC50  |  Pearson r: {corr:.4f}')
print('★ = best for that metric')


## 9 · Download Combined Results

In [ ]:
!zip -j /content/cliffforge_combined_results.zip \
    /content/results_mmp/summary_mmp_combined.json \
    /content/chemprop_mmp_preds.csv

from google.colab import files as colab_files
colab_files.download('/content/cliffforge_combined_results.zip')
print('Download triggered.')


---
## Done

**Outputs:**
- `/content/results_mmp/summary_mmp_combined.json` — all three models in one file
- `/content/chemprop_mmp_preds.csv` — per-molecule ChemProp pIC50 predictions

**What to look for in the table:**
- **Direction accuracy on cliff pairs** — does ChemProp match CliffForge's 96.8%?
- **Lead-opt AP** — in the original, ChemProp got 0.570 and CliffForge got 0.466; now CliffForge is at 0.683
- **Within-series ρ** — in the original, ChemProp was 0.192 vs CliffForge 0.581